# Project 2: Handwritten Digit Recognizer (MNIST)
**Goal:** Train a convolutional neural network (CNN) to classify handwritten digits from 0 to 9.

**Dataset:** MNIST, downloaded automatically through TensorFlow/Keras.
**Evaluation:** Test accuracy, classification report, confusion matrix, and example predictions.
**Compute:** Designed to run on a CPU; a GPU can make training faster.

## 1. Install dependencies
TensorFlow is a larger package. Install it in the Python environment used by Jupyter. If imports fail after installation, restart the kernel.

In [ ]:
%pip install -q tensorflow matplotlib scikit-learn

## 2. Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

## 3. Load the MNIST dataset

In [ ]:
# Keras downloads MNIST automatically the first time this runs.
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)

## 4. View sample images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")
plt.suptitle("Sample MNIST training images")
plt.tight_layout()
plt.show()

## 5. Prepare the images
Pixel values range from 0 to 255. Scale them to 0–1 and add a channel dimension because the CNN expects image tensors shaped like `(height, width, channels)`.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print("CNN input shape:", x_train.shape[1:])

## 6. Build and train a CNN
The model uses two convolution layers, max pooling, dropout, and a dense output layer with 10 classes.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dropout(0.30),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.10,
    verbose=1
)

## 7. Plot training progress

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN training progress")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN loss")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Evaluate on the test set

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4%}")

probabilities = model.predict(x_test, verbose=0)
y_pred = np.argmax(probabilities, axis=1)

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, cmap="Blues", values_format="d"
)
plt.title("MNIST CNN — Test Set Confusion Matrix")
plt.tight_layout()
plt.show()

## 9. Visualize predictions

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    image_index = i
    predicted = int(y_pred[image_index])
    actual = int(y_test[image_index])
    ax.imshow(x_test[image_index].squeeze(), cmap="gray")
    ax.set_title(f"Pred: {predicted} | True: {actual}")
    ax.axis("off")
plt.suptitle("Example test-set predictions")
plt.tight_layout()
plt.show()

## 10. Predict a single test image
Change `image_index` to inspect another image. The displayed confidence is the model's probability for its selected class, not a guarantee of correctness.

In [ ]:
image_index = 0
image_to_predict = x_test[image_index]
prediction_scores = model.predict(image_to_predict[np.newaxis, ...], verbose=0)[0]
predicted_digit = int(np.argmax(prediction_scores))

plt.imshow(image_to_predict.squeeze(), cmap="gray")
plt.title(
    f"Predicted: {predicted_digit} | True: {int(y_test[image_index])} | "
    f"Confidence: {prediction_scores[predicted_digit]:.1%}"
)
plt.axis("off")
plt.show()

print("Probabilities for digits 0–9:")
print(np.round(prediction_scores, 3))

## 11. Save the model (optional)

In [ ]:
model.save("mnist_digit_recognizer.keras")
print("Saved: mnist_digit_recognizer.keras")

## 12. What to write in your project report
- **Problem:** Recognize handwritten digits from image pixels.
- **Dataset:** MNIST grayscale images, each 28×28 pixels.
- **Preprocessing:** Normalize pixel values and add a channel dimension.
- **Algorithm:** Convolutional neural network (CNN).
- **Evaluation:** Test accuracy, per-class precision/recall/F1, confusion matrix and sample predictions.
- **Limitation:** MNIST images are small and centered; real-world photos or differently written digits may be harder to classify.